# 1. Analysis - Relative value

This notebook compares the current Single Asset Profile ticker against a peer set using trailing valuation multiples and yield metrics from Financial Modeling Prep. It follows the same shared `params.py` pattern as the other Valuation notebooks and is organized into setup, peer-universe construction, peer-metric collection, and relative-value diagnostics.

Run the notebook from top to bottom after updating `ticker_str` in `../../params.py`. If the automatic peer screen is not good enough for the company you are analyzing, update `manual_peer_symbols` and the peer filter settings in Cell 4 before rerunning the downstream cells.

In [ ]:
# 2. Setup and FMP helpers
from pathlib import Path
import os
import runpy

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import requests


PARAMS_FILE_CANDIDATES = []
for base_path in [Path.cwd(), *Path.cwd().parents]:
    PARAMS_FILE_CANDIDATES.extend(
        [
            base_path / "params.py",
            base_path / "Single Asset Profile" / "params.py",
            base_path / "Notebooks" / "Single Asset Profile" / "params.py",
            base_path / "Investment Scripts" / "Notebooks" / "Single Asset Profile" / "params.py",
        ]
    )

for params_file in PARAMS_FILE_CANDIDATES:
    if params_file.exists():
        break
else:
    raise FileNotFoundError("Could not locate params.py in the Single Asset Profile workspace.")

notebook_params = runpy.run_path(str(params_file))
single_asset_params = notebook_params["get_single_asset_profile_params"]()

TICKER = single_asset_params["ticker_str"]
BASE_URL = "https://financialmodelingprep.com/stable"
PEER_LIMIT = 12


def find_project_root(start_path: Path | None = None) -> Path:
    current = (start_path or Path.cwd()).resolve()
    candidates = [current, *current.parents]
    for candidate in candidates:
        if (candidate / "pyproject.toml").exists() or (candidate / ".git").exists():
            return candidate
    return current


def load_project_env() -> None:
    env_path = find_project_root() / ".env"
    if not env_path.exists():
        return

    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue

        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip()
        if value and value[0] == value[-1] and value[0] in {'"', "'"}:
            value = value[1:-1]
        os.environ.setdefault(key, value)


def get_api_key() -> str:
    load_project_env()
    api_key = os.getenv("FMP_API_KEY") or os.getenv("FINANCIAL_MODELING_PREP_API_KEY")
    if api_key:
        return api_key
    raise KeyError("Missing FMP_API_KEY in the project .env file or environment.")


def normalize_symbol(ticker: str) -> str:
    normalized = str(ticker).strip().upper()
    for old, new in ((chr(92), "."), ("/", "."), ("-", ".")):
        normalized = normalized.replace(old, new)
    return normalized


def request_json(endpoint: str, **params):
    response = requests.get(
        f"{BASE_URL}/{endpoint}",
        params={**params, "apikey": get_api_key()},
        timeout=30,
    )
    response.raise_for_status()
    payload = response.json()
    if isinstance(payload, dict) and payload.get("Error Message"):
        raise RuntimeError(payload["Error Message"])
    return payload


def safe_first_record(endpoint: str, **params) -> dict:
    payload = request_json(endpoint, **params)
    if isinstance(payload, list):
        return payload[0] if payload else {}
    if isinstance(payload, dict):
        return payload
    return {}


def pick_numeric(record: dict, *keys: str) -> float:
    for key in keys:
        if not isinstance(record, dict):
            continue
        value = record.get(key)
        if value in (None, "", "None"):
            continue
        numeric_value = pd.to_numeric(pd.Series([value]), errors="coerce").iloc[0]
        if pd.notna(numeric_value):
            return float(numeric_value)
    return np.nan


def apply_standard_figure_layout(fig: go.Figure, title: str, height: int, bottom_margin: int = 110) -> None:
    fig.update_layout(
        title={"text": title, "x": 0.5, "xanchor": "center"},
        template="plotly_dark",
        paper_bgcolor="#020817",
        plot_bgcolor="#0f172a",
        font={"color": "#e2e8f0"},
        hovermode="x unified",
        hoverlabel={"bgcolor": "#0f172a", "font": {"color": "#e2e8f0", "size": 13}, "namelength": -1},
        legend={"orientation": "h", "yanchor": "top", "y": -0.18, "x": 0, "xanchor": "left"},
        autosize=True,
        height=height,
        margin={"l": 60, "r": 30, "t": 100, "b": bottom_margin},
    )


SYMBOL = normalize_symbol(TICKER)
SYMBOL

## 3. Build the Peer Universe

This cell loads company context, pulls peer symbols from Financial Modeling Prep's Stock Peer Comparison API, and then filters that raw peer set with configurable sector, industry, exchange, and market-cap rules. The target ticker is always forced into the peer list, and `manual_peer_symbols` can override the filtered set when needed.

In [ ]:
# 4. Load company context and peer candidates
quote_snapshot = safe_first_record("quote", symbol=SYMBOL)
profile_snapshot = safe_first_record("profile", symbol=SYMBOL)

company_name = quote_snapshot.get("name") or profile_snapshot.get("companyName") or SYMBOL
industry_name = profile_snapshot.get("industry") or quote_snapshot.get("industry") or ""
sector_name = profile_snapshot.get("sector") or quote_snapshot.get("sector") or ""
target_exchange = (
    profile_snapshot.get("exchangeShortName")
    or profile_snapshot.get("exchange")
    or quote_snapshot.get("exchange")
    or quote_snapshot.get("exchangeShortName")
    or ""
)
target_market_cap = pick_numeric(quote_snapshot, "marketCap", "marketCapTTM")

manual_peer_symbols = []
manual_peer_symbols = [normalize_symbol(symbol) for symbol in manual_peer_symbols if str(symbol).strip()]

require_same_sector = True
require_same_industry = False
require_same_exchange = False
market_cap_lower_multiple = 0.2
market_cap_upper_multiple = 5.0

peer_response = []
try:
    peer_response = request_json("stock-peers", symbol=SYMBOL)
except Exception as exc:
    print(f"Peer API unavailable: {exc}")

raw_peer_symbols = []
peer_candidates = peer_response if isinstance(peer_response, list) else [peer_response]
for record in peer_candidates:
    if isinstance(record, str):
        candidate_values = [record]
    elif isinstance(record, dict):
        candidate_values = []
        for key in ("peersList", "peers", "symbols", "companies"):
            value = record.get(key)
            if isinstance(value, list):
                candidate_values.extend(value)
        for key in ("symbol", "peerSymbol"):
            value = record.get(key)
            if value:
                candidate_values.append(value)
    else:
        candidate_values = []

    for candidate in candidate_values:
        candidate_symbol = normalize_symbol(candidate)
        if candidate_symbol and candidate_symbol != SYMBOL:
            raw_peer_symbols.append(candidate_symbol)

raw_peer_symbols = list(dict.fromkeys(raw_peer_symbols))

filtered_peer_symbols = []
peer_filter_rows = []
for candidate_symbol in raw_peer_symbols:
    try:
        candidate_quote = safe_first_record("quote", symbol=candidate_symbol)
        candidate_profile = safe_first_record("profile", symbol=candidate_symbol)
    except Exception as exc:
        peer_filter_rows.append({
            "symbol": candidate_symbol,
            "sector": None,
            "industry": None,
            "exchange": None,
            "marketCapBn": np.nan,
            "sameSector": False,
            "sameIndustry": False,
            "sameExchange": False,
            "withinMarketCapBand": False,
            "passesFilters": False,
            "note": f"Lookup failed: {exc}",
        })
        continue

    candidate_sector = candidate_profile.get("sector") or candidate_quote.get("sector") or ""
    candidate_industry = candidate_profile.get("industry") or candidate_quote.get("industry") or ""
    candidate_exchange = (
        candidate_profile.get("exchangeShortName")
        or candidate_profile.get("exchange")
        or candidate_quote.get("exchange")
        or candidate_quote.get("exchangeShortName")
        or ""
)
    candidate_market_cap = pick_numeric(candidate_quote, "marketCap", "marketCapTTM")

    same_sector = (not require_same_sector) or (not sector_name) or (candidate_sector == sector_name)
    same_industry = (not require_same_industry) or (not industry_name) or (candidate_industry == industry_name)
    same_exchange = (not require_same_exchange) or (not target_exchange) or (candidate_exchange == target_exchange)
    within_market_cap_band = True
    if pd.notna(target_market_cap) and target_market_cap > 0 and pd.notna(candidate_market_cap):
        within_market_cap_band = (
            target_market_cap * market_cap_lower_multiple <= candidate_market_cap <= target_market_cap * market_cap_upper_multiple
        )

    passes_filters = same_sector and same_industry and same_exchange and within_market_cap_band
    peer_filter_rows.append({
        "symbol": candidate_symbol,
        "sector": candidate_sector,
        "industry": candidate_industry,
        "exchange": candidate_exchange,
        "marketCapBn": candidate_market_cap / 1_000_000_000 if pd.notna(candidate_market_cap) else np.nan,
        "sameSector": same_sector,
        "sameIndustry": same_industry,
        "sameExchange": same_exchange,
        "withinMarketCapBand": within_market_cap_band,
        "passesFilters": passes_filters,
        "note": "",
    })

    if passes_filters:
        filtered_peer_symbols.append(candidate_symbol)

peer_filter_summary = pd.DataFrame(peer_filter_rows)
selection_note = "Applied configured peer filters."
if not filtered_peer_symbols and raw_peer_symbols:
    filtered_peer_symbols = raw_peer_symbols.copy()
    selection_note = "Filters returned no peers, so the notebook fell back to the raw FMP peer list. Loosen the filter settings or add manual peers if needed."

peer_symbols = [SYMBOL, *manual_peer_symbols, *filtered_peer_symbols]
peer_symbols = [symbol for symbol in dict.fromkeys(peer_symbols) if symbol][:PEER_LIMIT]

peer_overview = pd.DataFrame([
    {
        "company": company_name,
        "symbol": SYMBOL,
        "sector": sector_name,
        "industry": industry_name,
        "exchange": target_exchange,
        "rawPeerCount": len(raw_peer_symbols),
        "filteredPeerCount": len(filtered_peer_symbols),
        "peerCount": len(peer_symbols),
        "peerSource": "FMP stock-peers endpoint",
        "selectionNote": selection_note,
    }
])

peer_overview

In [ ]:
# 5. Collect peer valuation metrics
metric_rows = []
for peer_symbol in peer_symbols:
    peer_quote = safe_first_record("quote", symbol=peer_symbol)
    peer_profile = safe_first_record("profile", symbol=peer_symbol)
    peer_ratios = safe_first_record("ratios-ttm", symbol=peer_symbol)
    peer_key_metrics = safe_first_record("key-metrics-ttm", symbol=peer_symbol)

    metric_rows.append({
        "symbol": peer_symbol,
        "companyName": peer_quote.get("name") or peer_profile.get("companyName") or peer_symbol,
        "sector": peer_profile.get("sector"),
        "industry": peer_profile.get("industry"),
        "price": pick_numeric(peer_quote, "price"),
        "marketCapBn": pick_numeric(peer_quote, "marketCap") / 1_000_000_000,
        "peRatio": pick_numeric(peer_ratios, "peRatioTTM", "priceEarningsRatioTTM", "peRatio"),
        "priceToSales": pick_numeric(peer_ratios, "priceToSalesRatioTTM", "priceToSalesRatio"),
        "priceToBook": pick_numeric(peer_ratios, "priceToBookRatioTTM", "priceToBookRatio"),
        "evToRevenue": pick_numeric(peer_key_metrics, "enterpriseValueOverRevenueTTM", "enterpriseValueOverRevenue"),
        "evToEbitda": pick_numeric(peer_key_metrics, "enterpriseValueOverEBITDATTM", "enterpriseValueOverEBITDA"),
        "freeCashFlowYieldPct": pick_numeric(peer_key_metrics, "freeCashFlowYieldTTM", "freeCashFlowYield") * 100,
        "dividendYieldPct": pick_numeric(peer_ratios, "dividendYielTTM", "dividendYieldTTM", "dividendYieldPercentageTTM", "dividendYield") * 100,
    })

peer_metrics = pd.DataFrame(metric_rows).drop_duplicates(subset=["symbol"]).reset_index(drop=True)
numeric_columns = [
    "price",
    "marketCapBn",
    "peRatio",
    "priceToSales",
    "priceToBook",
    "evToRevenue",
    "evToEbitda",
    "freeCashFlowYieldPct",
    "dividendYieldPct",
]
for column in numeric_columns:
    peer_metrics[column] = pd.to_numeric(peer_metrics[column], errors="coerce")

peer_metrics = peer_metrics.sort_values(["marketCapBn", "symbol"], ascending=[False, True]).reset_index(drop=True)
peer_metrics

In [ ]:
# 6. Summarize relative value vs peer medians
target_rows = peer_metrics.loc[peer_metrics["symbol"].eq(SYMBOL)]
if target_rows.empty:
    raise ValueError(f"Target ticker {SYMBOL} was not found in the peer metric table.")

target_row = target_rows.iloc[0]
metrics_to_compare = {
    "peRatio": "P/E",
    "priceToSales": "Price / Sales",
    "priceToBook": "Price / Book",
    "evToRevenue": "EV / Revenue",
    "evToEbitda": "EV / EBITDA",
    "freeCashFlowYieldPct": "FCF Yield (%)",
    "dividendYieldPct": "Dividend Yield (%)",
}

relative_value_rows = []
for column, label in metrics_to_compare.items():
    peer_series = peer_metrics.loc[peer_metrics["symbol"].ne(SYMBOL), column].dropna()
    target_value = pd.to_numeric(pd.Series([target_row[column]]), errors="coerce").iloc[0]
    if pd.isna(target_value) or peer_series.empty:
        continue

    peer_median = float(peer_series.median())
    is_yield_metric = column.endswith("YieldPct")
    if peer_median == 0:
        discount_to_peer_median_pct = np.nan
    elif is_yield_metric:
        discount_to_peer_median_pct = (target_value / peer_median - 1) * 100
    else:
        discount_to_peer_median_pct = (1 - target_value / peer_median) * 100

    cheaper_than_peers_pct = (peer_series <= target_value).mean() * 100 if is_yield_metric else (peer_series >= target_value).mean() * 100

    relative_value_rows.append({
        "metric": label,
        "targetValue": round(float(target_value), 2),
        "peerMedian": round(peer_median, 2),
        "discountToPeerMedianPct": round(discount_to_peer_median_pct, 2) if pd.notna(discount_to_peer_median_pct) else np.nan,
        "cheaperThanPeersPct": round(cheaper_than_peers_pct, 2),
    })

relative_value_summary = pd.DataFrame(relative_value_rows)
relative_value_summary

In [ ]:
# 7. Plot the target discount or premium vs peer medians
if relative_value_summary.empty:
    raise ValueError("No relative value metrics were available to plot.")

bar_colors = ["#22c55e" if value >= 0 else "#ef4444" for value in relative_value_summary["discountToPeerMedianPct"]]
relative_value_plot = go.Figure()
relative_value_plot.add_trace(
    go.Bar(
        x=relative_value_summary["metric"],
        y=relative_value_summary["discountToPeerMedianPct"],
        marker={"color": bar_colors},
        text=relative_value_summary["discountToPeerMedianPct"].map(lambda value: f"{value:.1f}%"),
        textposition="outside",
        name="Discount to peer median",
    )
)
relative_value_plot.add_hline(y=0, line_dash="dot", line_color="#94a3b8")
apply_standard_figure_layout(
    relative_value_plot,
    title=f"{company_name} relative valuation vs peer medians",
    height=620,
    bottom_margin=120,
)
relative_value_plot.update_xaxes(title="Metric", tickangle=-20)
relative_value_plot.update_yaxes(title="Discount to peer median (%)")
relative_value_plot.show(config={"responsive": True, "displaylogo": False})